# DistribuMTY · PSO vs ACO en el problema del agente viajero (TSP) sobre una red vial
**Proyecto integrador – Algoritmos bioinspirados**

Este notebook:
1. Construye una red vial dirigida (calles de un solo sentido, avenidas rápidas y calles lentas).
2. Calcula las matrices de distancia (km) y tiempo (min) con Dijkstra.
3. Obtiene el **óptimo exacto** (Held–Karp) y la heurística del **vecino más cercano** como referencias.
4. Compara **PSO** (random keys), **ACO** (Ant System elitista) y los híbridos **+2-opt** con búsqueda en malla de hiperparámetros (15 corridas por configuración).
5. Genera tablas, gráficas y `resultados.json` (lo usa la página web de demostración).

**Fuente de la red:** `USAR_OSM = True` descarga las calles reales de OpenStreetMap (radio de 1.1 km alrededor de la Macroplaza, Monterrey).
Con `USAR_OSM = False` usa una ciudad sintética reproducible (corre sin internet).


In [ ]:
# Instalación (solo Colab)
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx"])


In [ ]:
# Configuración
import os, time, json, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

USAR_OSM  = True 
LUGAR     = "Centro, Monterrey, Nuevo León, México"
SEMILLA   = 7       # semilla de la red y de las paradas
N_PUNTOS  = 15      # 1 depósito + 14 paradas
ITERS     = 200
CORRIDAS  = 15
OUT = "resultados"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
C_ACO, C_PSO, C_HIB, C_REF = "#7c3aed", "#0d9488", "#ea580c", "#475569"


## 1. Red vial dirigida
Nodos = cruces, aristas = tramos de calle con longitud (km) y velocidad (km/h).
Si una calle es de un solo sentido la arista existe solo en esa dirección. Se conserva la componente fuertemente conexa más grande.


In [ ]:
# Construcción de la red
def red_sintetica(seed=SEMILLA, n=13, paso=0.16):
    rng = np.random.default_rng(seed)
    xy = np.array([[j*paso, i*paso] for i in range(n) for j in range(n)]) + rng.normal(0, 0.02, (n*n, 2))
    # cada calle (fila o columna): avenida cada 4 líneas (doble sentido, 50 km/h); el resto 25-35 km/h
    def calle(k):
        if k % 4 == 0: return 0, 50.0
        return int(rng.choice([0, 1, -1], p=[.35, .325, .325])), float(rng.choice([25, 30, 35]))
    filas  = [calle(k) for k in range(n)]
    cols   = [calle(k) for k in range(n)]
    E = []
    def add(a, b, sentido, vel):
        d = float(np.linalg.norm(xy[a]-xy[b]))
        if sentido >= 0: E.append((a, b, d, vel))
        if sentido <= 0: E.append((b, a, d, vel))
    for i in range(n):
        for j in range(n):
            if j+1 < n: add(i*n+j, i*n+j+1, *filas[i])
            if i+1 < n: add(i*n+j, (i+1)*n+j, *cols[j])
    # quitar ~6% de tramos sin romper la conectividad fuerte
    N = n*n
    def fuerte(E):
        A = csr_matrix(([1]*len(E), ([e[0] for e in E], [e[1] for e in E])), shape=(N, N))
        return connected_components(A, directed=True, connection="strong")[0] == 1
    for k in rng.permutation(len(E))[:int(0.06*len(E))]:
        trial = [e for q, e in enumerate(E) if q != k]
        if fuerte(trial): E = trial
    nombres = [f"Calle {i+1} y Av. {chr(65+j)}" if j % 4 == 0 else f"Calle {i+1} y Calle {chr(65+j)}"
               for i in range(n) for j in range(n)]
    return xy, E, nombres

LATLON = None   # latitud/longitud de cada nodo (para dibujar sobre el mapa real)
def red_osm(lugar=None, centro=(25.6694, -100.3099), radio=1100):
    """Descarga las calles en un cuadro de ±radio metros alrededor de `centro` (Macroplaza, Monterrey)."""
    global LATLON
    import osmnx as ox
    G = ox.graph_from_point(centro, dist=radio, network_type="drive")
    G = ox.add_edge_speeds(G)
    try:    G = ox.truncate.largest_component(G, strongly=True)
    except Exception: G = ox.utils_graph.get_largest_component(G, strongly=True)
    ids = {u: k for k, u in enumerate(G.nodes)}
    LATLON = np.array([[d["y"], d["x"]] for _, d in G.nodes(data=True)])
    lat0 = LATLON[:, 0].mean()
    xy = np.array([[d["x"]*111.32*np.cos(np.radians(lat0)), d["y"]*110.57] for _, d in G.nodes(data=True)])
    xy -= xy.min(0)
    best, calles = {}, {k: set() for k in ids.values()}
    for u, v, d in G.edges(data=True):
        n = d.get("name"); n = n[0] if isinstance(n, list) else n
        if n: calles[ids[u]].add(n); calles[ids[v]].add(n)
        key = (ids[u], ids[v]); c = (d["length"]/1000, d.get("speed_kph", 30))
        if key not in best or c[0] < best[key][0]: best[key] = c
    E = [(a, b, l, s) for (a, b), (l, s) in best.items()]
    nombres = [" y ".join(sorted(calles[k])[:2]) or f"Cruce {k}" for k in range(len(ids))]
    return xy, E, nombres

xy, E, nombres = red_osm(LUGAR) if USAR_OSM else red_sintetica()
# componente fuertemente conexa más grande y reindexado
N0 = len(xy)
A0 = csr_matrix(([1]*len(E), ([e[0] for e in E], [e[1] for e in E])), shape=(N0, N0))
_, lab = connected_components(A0, directed=True, connection="strong")
mayor = np.bincount(lab).argmax(); keep = np.where(lab == mayor)[0]; remap = {int(o): k for k, o in enumerate(keep)}
xy = xy[keep]; nombres = [nombres[o] for o in keep]
E = [(remap[a], remap[b], l, s) for a, b, l, s in E if a in remap and b in remap]
N = len(xy)
if LATLON is not None: LATLON = LATLON[keep]
print(f"Red final: {N} nodos, {len(E)} tramos dirigidos, {sum(1 for e in E if (e[1],e[0]) not in {(q[0],q[1]) for q in E})} de un solo sentido")


## 2. Paradas y matrices de distancia / tiempo (Dijkstra)


In [ ]:
# Paradas y matrices
def elegir_puntos(xy, k=N_PUNTOS, sep=0.45, seed=SEMILLA):
    rng = np.random.default_rng(seed)
    dep = int(np.argmin(np.linalg.norm(xy - xy.mean(0), axis=1)))   # depósito: cruce más céntrico
    pts = [dep]
    for c in rng.permutation(len(xy)):
        if all(np.linalg.norm(xy[c]-xy[p]) >= sep for p in pts): pts.append(int(c))
        if len(pts) == k: break
    return pts
PTS = elegir_puntos(xy)

def matrices(E, N, pts):
    L = csr_matrix(([e[2] for e in E], ([e[0] for e in E], [e[1] for e in E])), shape=(N, N))
    T = csr_matrix(([e[2]/e[3]*60 for e in E], ([e[0] for e in E], [e[1] for e in E])), shape=(N, N))
    dL, pL = dijkstra(L, directed=True, indices=pts, return_predecessors=True)
    dT, pT = dijkstra(T, directed=True, indices=pts, return_predecessors=True)
    return dL[:, pts], dT[:, pts], pL, pT
D, Tm, PL, PT = matrices(E, N, PTS)
asim = np.abs(D - D.T).max()
print(f"Matriz {D.shape}, asimetría máx = {asim:.2f} km")
pd.DataFrame([[i, nombres[p], *xy[p].round(3)] for i, p in enumerate(PTS)],
             columns=["Punto", "Ubicación", "x_km", "y_km"]).to_csv(f"{OUT}/puntos.csv", index=False)


## 3. Funciones comunes: costo, 2-opt, Held–Karp (óptimo), vecino más cercano


In [ ]:
# Utilidades
def costos(M, tours):
    """tours: (P, n-1) con valores 1..n-1 (el depósito 0 es inicio y fin)."""
    z = np.zeros((tours.shape[0], 1), dtype=int)
    f = np.hstack([z, tours, z])
    return M[f[:, :-1], f[:, 1:]].sum(1)

def _pares(m):
    return np.array([(i, j) for i in range(m-1) for j in range(i+1, m)])

def dos_opt(M, tours):
    """2-opt por mejor mejora, en lote. Invierte tramos mientras reduzca el costo (válido para matrices asimétricas)."""
    tours = tours.copy(); m = tours.shape[1]; pr = _pares(m)
    idx = np.tile(np.arange(m), (len(pr), 1))
    for r, (i, j) in enumerate(pr): idx[r, i:j+1] = np.arange(j, i-1, -1)
    f = costos(M, tours)
    while True:
        cand = tours[:, idx]                                   # (P, pares, m)
        fc = costos(M, cand.reshape(-1, m)).reshape(len(tours), -1)
        b = fc.argmin(1); mejora = fc[np.arange(len(tours)), b] < f - 1e-12
        if not mejora.any(): return tours, f
        tours[mejora] = cand[np.arange(len(tours)), b][mejora]
        f = np.where(mejora, fc[np.arange(len(tours)), b], f)

def held_karp(M):
    n = len(M); m = n-1; S_ = 1 << m
    dp = np.full((S_, m), np.inf); par = np.full((S_, m), -1, dtype=int)
    for j in range(m): dp[1 << j, j] = M[0, j+1]
    for S in range(1, S_):
        for j in range(m):
            if not (S >> j) & 1 or not np.isfinite(dp[S, j]): continue
            for k in range(m):
                if (S >> k) & 1: continue
                v = dp[S, j] + M[j+1, k+1]; T = S | (1 << k)
                if v < dp[T, k]: dp[T, k] = v; par[T, k] = j
    full = S_-1; fin = dp[full] + M[1:, 0]; j = int(fin.argmin()); best = fin[j]
    tour = []; S = full
    while j != -1:
        tour.append(j+1); pj = par[S, j]; S ^= (1 << j); j = pj
    return best, np.array(tour[::-1])

def vecino_mas_cercano(M):
    n = len(M); vis = {0}; t = []; cur = 0
    while len(vis) < n:
        c = min((j for j in range(n) if j not in vis), key=lambda j: M[cur, j]); t.append(c); vis.add(c); cur = c
    return costos(M, np.array([t]))[0], np.array(t)

t0 = time.time()
OPT_D, RUTA_OPT_D = held_karp(D); OPT_T, RUTA_OPT_T = held_karp(Tm)
NN_D, RUTA_NN = vecino_mas_cercano(D)
print(f"Óptimo distancia = {OPT_D:.3f} km | óptimo tiempo = {OPT_T:.2f} min | NN = {NN_D:.3f} km ({100*(NN_D/OPT_D-1):.1f}% sobre el óptimo) [{time.time()-t0:.1f}s]")


## 4. Algoritmos
**PSO** con *random keys*: cada partícula es un vector real de 14 números; `argsort` da el orden de visita.
`v = w·v + c1·r1·(pbest − x) + c2·r2·(gbest − x)`, `|v| ≤ 1.5`.

**ACO** (Ant System + hormigas elitistas): `p(i→j) ∝ τ^α · (1/d)^β`, evaporación ρ, depósito Q/L, y refuerzo e·Q/L a la mejor ruta global. Feromona **dirigida** (la matriz es asimétrica).

Las variantes `local=True` aplican 2-opt (híbridos).


In [ ]:
# PSO y ACO
def pso(M, P=50, w=0.9, c1=2.0, c2=2.0, iters=ITERS, seed=0, vmax=1.5, local=False):
    rng = np.random.default_rng(seed); m = len(M)-1
    x = rng.random((P, m)); v = rng.uniform(-.5, .5, (P, m)); ar = np.arange(P)[:, None]
    def decodificar(x):
        t = np.argsort(x, axis=1) + 1
        if local:
            t, f = dos_opt(M, t); x[ar, t-1] = np.arange(m)/m      # escribir de regreso el orden mejorado
        else: f = costos(M, t)
        return t, f
    t, f = decodificar(x); pb, pf = x.copy(), f.copy()
    g = f.argmin(); gb, gf, gt = x[g].copy(), f[g], t[g].copy(); hist = []
    for _ in range(iters):
        v = np.clip(w*v + c1*rng.random((P, m))*(pb-x) + c2*rng.random((P, m))*(gb-x), -vmax, vmax)
        x = x + v; t, f = decodificar(x)
        mej = f < pf; pb[mej] = x[mej]; pf[mej] = f[mej]
        g = f.argmin()
        if f[g] < gf: gf, gb, gt = f[g], x[g].copy(), t[g].copy()
        hist.append(gf)
    return gt, gf, np.array(hist)

def aco(M, A=30, alpha=0.5, beta=2.0, rho=0.5, iters=ITERS, seed=0, Q=1.0, e=5, local=False):
    rng = np.random.default_rng(seed); n = len(M); eta = 1/(M + np.eye(n))
    tau = np.full((n, n), 1/vecino_mas_cercano(M)[0]); gt, gf = None, np.inf; hist = []
    for _ in range(iters):
        Pm = tau**alpha * eta**beta
        vis = np.zeros((A, n), bool); vis[:, 0] = True; cur = np.zeros(A, int); tours = np.zeros((A, n-1), int)
        for s in range(n-1):
            w = Pm[cur]; w[vis] = 0; p = w/w.sum(1, keepdims=True)
            nxt = np.minimum((p.cumsum(1) < rng.random((A, 1))).sum(1), n-1)
            nxt = np.where(vis[np.arange(A), nxt], (w > 0).argmax(1), nxt)      # protección numérica
            tours[:, s] = nxt; vis[np.arange(A), nxt] = True; cur = nxt
        if local: tours, f = dos_opt(M, tours)
        else: f = costos(M, tours)
        b = f.argmin()
        if f[b] < gf: gf, gt = f[b], tours[b].copy()
        tau *= (1-rho)
        for tr, L in zip(tours, f):
            ruta = np.r_[0, tr, 0]; tau[ruta[:-1], ruta[1:]] += Q/L
        ruta = np.r_[0, gt, 0]; tau[ruta[:-1], ruta[1:]] += e*Q/gf
        tau = np.maximum(tau, 1e-12); hist.append(gf)
    return gt, gf, np.array(hist)


## 5. Experimento: búsqueda en malla (24 configuraciones PSO + 24 ACO) × 15 corridas


In [ ]:
# Experimento
def correr(fn, M, opt, cfg, corridas=CORRIDAS, **fijos):
    fs, its, tms, hs, ts = [], [], [], [], []
    for s in range(corridas):
        t0 = time.perf_counter(); t, f, h = fn(M, seed=s, **cfg, **fijos); tms.append((time.perf_counter()-t0)*1000)
        fs.append(f); its.append(int(np.argmax(h <= f + 1e-9)) + 1); hs.append(h); ts.append(t)
    fs = np.array(fs)
    return dict(**cfg, Mejor=fs.min(), Promedio=fs.mean(), Peor=fs.max(), Desv=fs.std(ddof=1),
                Brecha=100*(fs.mean()/opt-1), Optimo=int((fs <= opt+1e-6).sum()), IterMejor=float(np.mean(its)),
                ms=float(np.mean(tms))), fs, np.mean(hs, 0), np.std(hs, 0), ts[int(fs.argmin())]

malla_pso = [dict(P=a, w=b, c1=c, c2=d) for a, b, c, d in itertools.product([20, 50], [.4, .7, .9], [1., 2.], [1., 2.])]
malla_aco = [dict(A=a, alpha=b, beta=c, rho=d) for a, b, c, d in itertools.product([10, 30], [.5, 1., 2.], [2., 5.], [.1, .5])]
t0 = time.time()
R_pso = [correr(pso, D, OPT_D, c) for c in malla_pso]
R_aco = [correr(aco, D, OPT_D, c) for c in malla_aco]
df_pso = pd.DataFrame([r[0] for r in R_pso]); df_aco = pd.DataFrame([r[0] for r in R_aco])
df_pso.index = df_aco.index = range(1, 25)
df_pso.to_csv(f"{OUT}/tabla_pso.csv"); df_aco.to_csv(f"{OUT}/tabla_aco.csv")
print(f"Malla completa en {time.time()-t0:.0f}s")
print(df_pso.round(3).to_string()); print(df_aco.round(3).to_string())


In [ ]:
# Mejores configuraciones y comparación directa (distancia)
ip, ia = df_pso.Promedio.idxmin()-1, df_aco.Promedio.idxmin()-1
cfg_p, cfg_a = malla_pso[ip], malla_aco[ia]
print("Mejor PSO:", cfg_p, "| Mejor ACO:", cfg_a)
res = {"ACO": R_aco[ia], "PSO": R_pso[ip], "PSO + 2-opt": correr(pso, D, OPT_D, cfg_p, local=True),
       "ACO + 2-opt": correr(aco, D, OPT_D, cfg_a, local=True)}
comp = pd.DataFrame({k: v[0] for k, v in res.items()}).loc[["Mejor", "Promedio", "Peor", "Desv", "Brecha", "Optimo", "IterMejor", "ms"]]
comp.to_csv(f"{OUT}/comparacion_distancia.csv"); print(comp.round(3))


In [ ]:
# Experimento: minimizar tiempo
res_t = {"ACO": correr(aco, Tm, OPT_T, cfg_a), "PSO": correr(pso, Tm, OPT_T, cfg_p),
         "PSO + 2-opt": correr(pso, Tm, OPT_T, cfg_p, local=True), "ACO + 2-opt": correr(aco, Tm, OPT_T, cfg_a, local=True)}
comp_t = pd.DataFrame({k: v[0] for k, v in res_t.items()}).loc[["Mejor", "Promedio", "Peor", "Desv", "Brecha", "Optimo"]]
comp_t.to_csv(f"{OUT}/comparacion_tiempo.csv"); print(comp_t.round(3))


## 6. Gráficas


In [ ]:
# Gráficas estadísticas
col = {"ACO": C_ACO, "PSO": C_PSO, "PSO + 2-opt": C_HIB, "ACO + 2-opt": "#db2777"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, r in res.items():
    h, s = r[2], r[3]; ax.plot(h, color=col[k], label=f"{k} (promedio de {CORRIDAS} corridas)"); ax.fill_between(range(len(h)), h-s, h+s, color=col[k], alpha=.13)
ax.axhline(OPT_D, ls="--", c=C_REF, label=f"Óptimo exacto ({OPT_D:.2f} km)"); ax.axhline(NN_D, ls=":", c="gray", label=f"Vecino más cercano ({NN_D:.2f} km)")
ax.set_xlabel("Iteración"); ax.set_ylabel("Mejor distancia encontrada (km)"); ax.set_title("Convergencia de las mejores configuraciones"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_ylim(OPT_D-.5, NN_D+3); fig.tight_layout(); fig.savefig(f"{OUT}/fig1_convergencia.png"); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5)); ks = list(res)
bp = ax.boxplot([res[k][1] for k in ks], patch_artist=True, tick_labels=ks)
for b, k in zip(bp["boxes"], ks): b.set_facecolor(col[k]); b.set_alpha(.4)
for i, k in enumerate(ks): ax.scatter(np.random.default_rng(1).normal(i+1, .04, CORRIDAS), res[k][1], s=14, c=col[k], zorder=3)
ax.axhline(OPT_D, ls="--", c=C_REF); ax.set_ylabel("Distancia final (km)"); ax.set_title(f"Distribución de las {CORRIDAS} corridas"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_boxplot.png"); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, df, c, t in [(axs[0], df_pso, C_PSO, "PSO"), (axs[1], df_aco, C_ACO, "ACO")]:
    ax.errorbar(df.index, df.Promedio, df.Desv, fmt="o", c=c, capsize=3, ms=4); ax.axhline(OPT_D, ls="--", c=C_REF)
    ax.set_title(f"{t}: promedio ± desv. por configuración"); ax.set_xlabel("Configuración"); ax.grid(alpha=.3)
axs[0].set_ylabel("Distancia (km)"); fig.tight_layout(); fig.savefig(f"{OUT}/fig3_configs.png"); plt.show()

fig, axs = plt.subplots(2, 4, figsize=(13, 5.5)); efectos = {}
for r, (df, hp, c, nom) in enumerate([(df_pso, ["P", "w", "c1", "c2"], C_PSO, "PSO"), (df_aco, ["A", "alpha", "beta", "rho"], C_ACO, "ACO")]):
    for q, h in enumerate(hp):
        g = df.groupby(h).Promedio.mean(); efectos[f"{nom}:{h}"] = {str(k): float(v) for k, v in g.items()}
        axs[r, q].bar(g.index.astype(str), g.values, color=c, alpha=.8); axs[r, q].set_title(f"{nom}: {h}", fontsize=9)
        axs[r, q].set_ylim(g.min()-.3, g.max()+.3)
        for i, v in enumerate(g.values): axs[r, q].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Distancia promedio por valor de cada hiperparámetro"); fig.tight_layout(); fig.savefig(f"{OUT}/fig4_hiperparametros.png"); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.8)); ks = list(res)
ax.bar(ks, [res[k][0]["ms"] for k in ks], color=[col[k] for k in ks])
for i, k in enumerate(ks): ax.text(i, res[k][0]["ms"], f"{res[k][0]['ms']:.0f} ms", ha="center", va="bottom")
ax.set_ylabel("ms por corrida"); ax.set_title(f"Tiempo de cómputo ({ITERS} iteraciones)"); fig.tight_layout(); fig.savefig(f"{OUT}/fig5_tiempo.png"); plt.show()


In [ ]:
# Mapas de rutas
def camino(pred, a, b):
    p = [PTS[b]]
    while p[-1] != PTS[a]: p.append(int(pred[a, p[-1]]))
    return p[::-1]
def polilinea(orden, pred):
    r = np.r_[0, orden, 0]; nodos = []
    for a, b in zip(r[:-1], r[1:]): nodos += camino(pred, a, b)[:-1]
    return nodos + [PTS[0]]
def dibujar(ax, orden, pred, color, titulo):
    for a, b, *_ in E: ax.plot(*zip(xy[a], xy[b]), c="#cbd5e1", lw=.5)
    pl = polilinea(orden, pred); ax.plot(xy[pl, 0], xy[pl, 1], c=color, lw=2.2, alpha=.9)
    for k, p in enumerate(PTS):
        ax.scatter(*xy[p], s=170 if k == 0 else 120, c="#16a34a" if k == 0 else "white", ec="k", zorder=4)
        ax.text(*xy[p], "D" if k == 0 else k, ha="center", va="center", fontsize=7, zorder=5)
    ax.set_title(titulo, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
def costo_t(orden, M): return costos(M, np.array([orden]))[0]
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
dibujar(axs[0], RUTA_OPT_D, PL, C_ACO, f"Mín. distancia: {OPT_D:.2f} km · {costo_t(RUTA_OPT_D, Tm):.1f} min")
dibujar(axs[1], RUTA_OPT_T, PT, C_HIB, f"Mín. tiempo: {costo_t(RUTA_OPT_T, D):.2f} km · {OPT_T:.1f} min")
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_rutas_dist_tiempo.png"); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
dibujar(axs[0], res["ACO"][4], PL, C_ACO, f"Mejor ruta ACO: {res['ACO'][0]['Mejor']:.2f} km")
dibujar(axs[1], RUTA_NN, PL, C_REF, f"Vecino más cercano: {NN_D:.2f} km · {costo_t(RUTA_NN, Tm):.1f} min")
fig.tight_layout(); fig.savefig(f"{OUT}/fig7_rutas_aco_nn.png"); plt.show()


In [ ]:
# Exportar resultados para la página web
def lim(v): return [round(float(a), 4) for a in v[::4]]
web = dict(
  graph=dict(nodes=np.round(xy, 3).tolist(), edges=[[a, b, round(l, 4), s] for a, b, l, s in E], names=nombres, stops=PTS,
                latlon=(np.round(LATLON, 6).tolist() if LATLON is not None else None)),
  ref=dict(opt_d=OPT_D, opt_t=OPT_T, nn_d=NN_D, ruta_opt_d=RUTA_OPT_D.tolist(), ruta_opt_t=RUTA_OPT_T.tolist(), ruta_nn=RUTA_NN.tolist()),
  mejores=dict(pso=cfg_p, aco=cfg_a),
  comparacion={k: {m: float(v) for m, v in comp[k].items()} for k in comp},
  comparacion_t={k: {m: float(v) for m, v in comp_t[k].items()} for k in comp_t},
  conv={k: lim(r[2]) for k, r in res.items()},
  tabla_pso=df_pso.round(3).reset_index().to_dict("records"), tabla_aco=df_aco.round(3).reset_index().to_dict("records"),
  efectos=efectos)
json.dump(web, open(f"{OUT}/resultados.json", "w"))
print("Listo. Archivos en", OUT)
